# Merge S3+S4 — Pseudo-label (inference chéo) + Ghép Dataset020_KneeUnion (8-class)

- **S3**: Model_iMorph → ảnh OAI-ZIB (pseudo meniscus+patella); Model_ZIB → ảnh iMorphics (pseudo bone).
- **S4**: ghép **GT + pseudo** (GT ưu tiên, pseudo chỉ điền voxel background) → `Dataset020_KneeUnion` nhãn 8-class.

Cùng domain OAI DESS nên pseudo-label đáng tin. Dùng `checkpoint_best.pth` (an toàn dù train chưa hết).


In [ ]:
!pip install -q nnunetv2


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 0) Cấu hình — CHỈNH trainer cho khớp cái bạn đã train


In [ ]:
import os
from pathlib import Path

os.environ["nnUNet_raw"]          = "/content/drive/MyDrive/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/content/nnUNet_preprocessed"
os.environ["nnUNet_results"]      = "/content/drive/MyDrive/nnUNet_results"

D1   = Path("/content/drive/MyDrive/nnUNet_raw/Dataset001_KneeOA")     # OAI-ZIB (GT+anh)
D12  = Path("/content/drive/MyDrive/nnUNet_raw/Dataset012_iMorphics")  # iMorphics (GT+anh, imagesTr=140 train)
UNION= Path("/content/drive/MyDrive/nnUNet_raw/Dataset020_KneeUnion")
(UNION/"imagesTr").mkdir(parents=True, exist_ok=True)
(UNION/"labelsTr").mkdir(parents=True, exist_ok=True)

PRED_MEN  = "/content/pred_meniscus_on_zib"     # Model_iMorph tren ZIB
PRED_BONE = "/content/pred_bone_on_imorph"      # Model_ZIB tren iMorph

# >>> CHINH cho dung trainer da train <<<
TR_ZIB    = "nnUNetTrainer_100epochs"
TR_IMORPH = "nnUNetTrainer_100epochs"   # doi thanh _250epochs neu ban train 250
CHK       = "checkpoint_best.pth"

# kiem model ton tai
for d, tr in [("Dataset001_KneeOA", TR_ZIB), ("Dataset013_iMorphSpec", TR_IMORPH)]:
    mp = Path(os.environ["nnUNet_results"])/d/f"{tr}__nnUNetResEncUNetLPlans__3d_fullres"/"fold_0"/CHK
    print(("OK  " if mp.exists() else "MISSING "), mp)


## 1) S3 — Inference chéo (pseudo-label)

`--disable_tta` cho nhanh (pseudo không cần TTA). ~404 ảnh ZIB + 140 ảnh iMorph.


In [ ]:
# Model_iMorph (d 13) -> anh OAI-ZIB  => pseudo meniscus + patella
!nnUNetv2_predict -i {D1}/imagesTr -o {PRED_MEN} -d 13 -c 3d_fullres -p nnUNetResEncUNetLPlans -tr {TR_IMORPH} -f 0 -chk {CHK} --disable_tta


In [ ]:
# Model_ZIB (d 1) -> anh iMorphics (train) => pseudo bone
!nnUNetv2_predict -i {D12}/imagesTr -o {PRED_BONE} -d 1 -c 3d_fullres -p nnUNetResEncUNetLPlans -tr {TR_ZIB} -f 0 -chk {CHK} --disable_tta


## 2) S4 — Ghép GT + pseudo → Dataset020 (8-class)

- ZIB case: GT (1–5) + pseudo **6,7,8** (từ specialist 4,5,6).
- iMorph case: GT (2,4,5,6,7,8) + pseudo **1,3**.
- Dùng SimpleITK để **geometry nhãn khớp ảnh tuyệt đối** (tránh lỗi spacing).


In [ ]:
import SimpleITK as sitk, numpy as np, shutil
from pathlib import Path

def merge_case(img_path, gt_path, pred_path, add_map, out_img, out_lbl):
    img_itk = sitk.ReadImage(str(img_path))
    gt = sitk.GetArrayFromImage(sitk.ReadImage(str(gt_path))).astype(np.uint8)
    pr = sitk.GetArrayFromImage(sitk.ReadImage(str(pred_path))).astype(np.uint8)
    merged = gt.copy()
    for src, dst in add_map.items():          # chi dien vao voxel background
        merged[(gt == 0) & (pr == src)] = dst
    out = sitk.GetImageFromArray(merged); out.CopyInformation(img_itk)
    sitk.WriteImage(out, str(out_lbl))
    shutil.copy(str(img_path), str(out_img))

# --- ZIB train: them meniscus+patella (specialist 4,5,6 -> union 6,7,8) ---
ADD_MEN = {4:6, 5:7, 6:8}
zib = sorted(p.name.replace(".nii.gz","") for p in (D1/"labelsTr").glob("*.nii.gz"))
for c in zib:
    merge_case(D1/"imagesTr"/f"{c}_0000.nii.gz", D1/"labelsTr"/f"{c}.nii.gz",
               Path(PRED_MEN)/f"{c}.nii.gz", ADD_MEN,
               UNION/"imagesTr"/f"{c}_0000.nii.gz", UNION/"labelsTr"/f"{c}.nii.gz")
print("ZIB merged:", len(zib))

# --- iMorph train: them bone (ZIB 1,3) ---
ADD_BONE = {1:1, 3:3}
imo = sorted(p.name.replace(".nii.gz","") for p in (D12/"labelsTr").glob("*.nii.gz"))
for c in imo:
    merge_case(D12/"imagesTr"/f"{c}_0000.nii.gz", D12/"labelsTr"/f"{c}.nii.gz",
               Path(PRED_BONE)/f"{c}.nii.gz", ADD_BONE,
               UNION/"imagesTr"/f"{c}_0000.nii.gz", UNION/"labelsTr"/f"{c}.nii.gz")
print("iMorph merged:", len(imo), "| tong:", len(zib)+len(imo))


## 3) dataset.json (8-class union)


In [ ]:
import json, glob
labels = {"background":0,"femoral_bone":1,"femoral_cartilage":2,"tibial_bone":3,
          "medial_tibial_cartilage":4,"lateral_tibial_cartilage":5,
          "medial_meniscus":6,"lateral_meniscus":7,"patellar_cartilage":8}
n = len(glob.glob(str(UNION/"imagesTr"/"*_0000.nii.gz")))
json.dump({"channel_names":{"0":"MRI"},"labels":labels,"numTraining":n,
           "file_ending":".nii.gz","name":"KneeUnion8",
           "description":"OAI-ZIB + iMorphics, pseudo-label completion, 8-class union."},
          open(UNION/"dataset.json","w"), indent=2, ensure_ascii=False)
print("numTraining:", n)


## 4) Verify + QC overlay (kiểm pseudo-label ghép đúng)


In [ ]:
import numpy as np, nibabel as nib, matplotlib.pyplot as plt, matplotlib.colors as mcolors, random
from matplotlib.patches import Patch
from pathlib import Path

lbls = sorted((UNION/"labelsTr").glob("*.nii.gz"))
print("so ca:", len(lbls))
for f in random.sample(lbls, 3):
    u = np.unique(np.asanyarray(nib.load(str(f)).dataobj).astype(int))
    print(f"  {f.name}: {u.tolist()}")   # ky vong tap con {0..8}

NAMES={1:"fem_bone",2:"fem_cart",3:"tib_bone",4:"med_tib",5:"lat_tib",6:"med_men",7:"lat_men",8:"patellar"}
COL={1:"#8c564b",2:"#1f77b4",3:"#e377c2",4:"#2ca02c",5:"#bcbd22",6:"#d62728",7:"#ff7f0e",8:"#17becf"}
def show(cid):
    img=np.asanyarray(nib.load(str(UNION/"imagesTr"/f"{cid}_0000.nii.gz")).dataobj).astype(float)
    lab=np.asanyarray(nib.load(str(UNION/"labelsTr"/f"{cid}.nii.gz")).dataobj).astype(int)
    # chon axis+slice nhieu nhan
    best=(-1,0,0)
    for ax in range(3):
        c=(lab>0).sum(axis=tuple(i for i in range(3) if i!=ax))
        if c.max()>best[0]: best=(c.max(),ax,int(c.argmax()))
    ax,z=best[1],best[2]
    ims=np.take(img,z,axis=ax); lbs=np.take(lab,z,axis=ax)
    ims=(ims-ims.min())/(np.ptp(ims)+1e-6); pres=[int(l) for l in np.unique(lbs) if l]
    plt.figure(figsize=(6,6)); plt.imshow(ims.T,cmap="gray",origin="lower")
    rgba=np.zeros(lbs.shape+(4,))
    for l in pres:
        m=lbs==l
        if m.any(): rgba[m,:3]=mcolors.to_rgb(COL[l]); rgba[m,3]=0.55
    plt.imshow(np.transpose(rgba,(1,0,2)),origin="lower")
    plt.legend(handles=[Patch(color=COL[l],label=f"{l} {NAMES[l]}") for l in pres],
               loc="upper right",fontsize=7); plt.title(cid); plt.axis("off"); plt.show()

# 1 ca ZIB (xem meniscus pseudo them vao) + 1 ca iMorph (xem bone pseudo)
show(sorted(p.name.replace(".nii.gz","") for p in (D1/"labelsTr").glob("*.nii.gz"))[0])
show(sorted(p.name.replace(".nii.gz","") for p in (D12/"labelsTr").glob("*.nii.gz"))[0])
